## 1. Data Setup

In [ ]:
import os, glob, random, json

DATA_DIR   = '/kaggle/input/datasets/zphilip/nougat-training-dataset-example'
OUTPUT_DIR = '/kaggle/working/vlm_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Explore dataset structure
for root, dirs, files in os.walk(DATA_DIR):
    for f in files[:3]:
        print(os.path.join(root, f))

In [ ]:
images = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.png'), recursive=True))
mmds   = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.mmd'), recursive=True))
print('Total images:', len(images), ' | Total markdowns:', len(mmds))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for i, ax in enumerate(axes):
    img = Image.open(images[i]).convert('RGB')
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Document Sample {i+1}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dataset_samples.png'), dpi=80)
plt.show()

# Show a markdown sample
with open(mmds[0]) as f:
    print(f.read()[:600])

In [ ]:
mmd_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in mmds}

pairs = []
for img_path in images:
    key = os.path.splitext(os.path.basename(img_path))[0]
    if key in mmd_lookup:
        pairs.append({'image': img_path, 'markdown': mmd_lookup[key]})

print('Valid image-markdown pairs:', len(pairs))

In [ ]:
random.seed(42)
random.shuffle(pairs)

split       = int(0.8 * len(pairs))
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

with open(os.path.join(OUTPUT_DIR, 'train_pairs.json'), 'w') as f:
    json.dump(train_pairs, f)
with open(os.path.join(OUTPUT_DIR, 'val_pairs.json'), 'w') as f:
    json.dump(val_pairs, f)

print('Train:', len(train_pairs), ' | Val:', len(val_pairs))

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers>=4.45.0', 'peft>=0.12.0',
                'bitsandbytes>=0.43.0', 'trl>=0.11.0', 'accelerate>=0.34.0',
                'qwen-vl-utils', 'datasets'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Dual-GPU (T4 × 2) — GPU enforcement & environment setup ──────────────────
import os, sys, warnings

# MUST be set before any CUDA context is created
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch

# Hard-fail if no GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Kaggle → Settings → Accelerator → GPU T4 x2"
    )

import torch.autograd.graph as _tag
_tag.set_warn_on_accumulate_grad_stream_mismatch(False)
warnings.filterwarnings('ignore', message='.*use_reentrant.*', category=UserWarning)
warnings.filterwarnings('ignore', message='.*GradScaler.*',   category=FutureWarning)

# ── Reset ALL GPUs — clears leftover allocations from previous notebook runs ──
NUM_GPUS = torch.cuda.device_count()
for g in range(NUM_GPUS):
    with torch.cuda.device(g):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(g)

print(f"GPUs detected: {NUM_GPUS}")
for g in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(g)
    free, total = torch.cuda.mem_get_info(g)
    print(f"  GPU {g}: {props.name}  total={total//1024**3} GB  free={free//1024**3} GB")

if NUM_GPUS < 2:
    print("WARNING: Only 1 GPU found — set Accelerator to 'GPU T4 x2' in Kaggle settings.")

# balanced_low_0 → fewer layers on GPU 0 (carries embedding + output head overhead)
DEVICE_MAP     = 'balanced_low_0' if NUM_GPUS > 1 else 'cuda:0'
PRIMARY_DEVICE = torch.device('cuda:0')

print(f"device_map     : {DEVICE_MAP}")
print(f"PRIMARY_DEVICE : {PRIMARY_DEVICE}")

# Warm-up: initialise CUDA context on every GPU
for g in range(NUM_GPUS):
    _ = torch.zeros(1, device=f'cuda:{g}')
    del _
print("CUDA contexts initialised on all GPUs.")
